<a href="https://colab.research.google.com/github/ElFriede143/FUNDAI-Laboratories-BENGCOLITA/blob/main/Lab3_Game_AI__Connet_Four_Bengcolita.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import math
import ipywidgets as widgets
from IPython.display import display

In [33]:
class ConnectFourGame:
    ROWS = 6
    COLS = 7
    EMPTY = 0
    RED = 1
    YELLOW = 2

    def __init__(self):
        self.board = [[self.EMPTY for _ in range(self.COLS)] for _ in range(self.ROWS)]
        self.current_player = self.RED

    def available_columns(self):
        return [c for c in range(self.COLS) if self.board[0][c] == self.EMPTY]

    def drop_piece(self, col):
        if col not in self.available_columns():
            return None

        for row in range(self.ROWS - 1, -1, -1):
            if self.board[row][col] == self.EMPTY:
                self.board[row][col] = self.current_player
                self.current_player = self.YELLOW if self.current_player == self.RED else self.RED
                return row

        return None

    def undo_piece(self, row, col):
        self.board[row][col] = self.EMPTY
        self.current_player = self.YELLOW if self.current_player == self.RED else self.RED

    def get_winner(self):
        directions = [(0, 1), (1, 0), (1, 1), (-1, 1)]

        for r in range(self.ROWS):
            for c in range(self.COLS):
                piece = self.board[r][c]

                if piece == self.EMPTY:
                    continue

                for dr, dc, in directions:
                    count = 1

                    for step in range(1, 4):
                        nr = r + dr * step
                        nc = c + dc * step

                        if 0 <= nr < self.ROWS and 0 <= nc < self.COLS and self.board[nr][nc] == piece:
                          count += 1
                        else:
                            break

                    if count >= 4:
                        return piece

        return None

    def is_draw(self):
        return self.get_winner() is None and len(self.available_columns()) == 0

    def is_terminal(self):
        return self.get_winner() is not None or len(self.available_columns()) == 0

    def utility(self):
        winner = self.get_winner()

        if winner == self.RED:
            return 100
        elif winner == self.YELLOW:
            return -100
        else:
            return 0

def score_window(window, piece, empty):
        opponent = ConnectFourGame.YELLOW if piece == ConnectFourGame.RED else ConnectFourGame.RED

        piece_count = window.count(piece)
        empty_count = window.count(empty)
        opponent_count = window.count(opponent)

        if piece_count == 4:
            return 100
        elif piece_count == 3 and empty_count == 1:
            return 5
        elif piece_count == 2 and empty_count == 2:
            return 2

        if opponent_count == 3 and empty_count == 1:
            return -8

        return 0

def evaluate_connect4(game, piece):
        score = 0
        board = game.board
        empty = game.EMPTY

        # Center column bonus
        center_col = 3
        center_count = sum(1 for r in range(game.ROWS) if board[r][center_col] == piece)
        score += center_count * 3

        # Horizontal windows
        for r in range(game.ROWS):
            for c in range(game.COLS - 3):
                window = [board[r][c + i] for i in range(4)]
                score += score_window(window, piece, empty)

        # Vertical windows
        for c in range(game.COLS):
            for r in range(game.ROWS - 3):
                window = [board[r + i][c] for i in range(4)]
                score += score_window(window, piece, empty)

        # Diagonal down-right
        for r in range(game.ROWS - 3):
            for c in range(game.COLS - 3):
                window = [board[r + i][c + i] for i in range(4)]
                score += score_window(window, piece, empty)

        # Diagonal up-right
        for r in range(3, game.ROWS):
            for c in range(game.COLS - 3):
                window = [board[r - i][c + i] for i in range(4)]
                score += score_window(window, piece, empty)

        return score

def evaluate_state_red(game):
        return evaluate_connect4(game, ConnectFourGame.RED)

def minimax_connect4(game, depth, alpha=-math.inf, beta=math.inf):
        if game.is_terminal():
            return game.utility(), None

        if depth == 0:
            return evaluate_state_red(game), None

        # RED is MAX
        if game.current_player == ConnectFourGame.RED:
            best_value = -math.inf
            best_col = None

            for col in game.available_columns():
                row = game.drop_piece(col)
                value, _ = minimax_connect4(game, depth - 1, alpha, beta)
                game.undo_piece(row, col)

                if value > best_value:
                    best_value = value
                    best_col = col

                alpha = max(alpha, best_value)

                if alpha >= beta:
                    break

            return best_value, best_col

        # YELLOW is MIN
        else:
            best_value = math.inf
            best_col = None

            for col in game.available_columns():
                row = game.drop_piece(col)
                value, _ = minimax_connect4(game, depth - 1, alpha, beta)
                game.undo_piece(row, col)

                if value < best_value:
                    best_value = value
                    best_col = col

                beta = min(beta, best_value)

                if alpha >= beta:
                    break

            return best_value, best_col
def connect4_board_html(game):
        symbols = {
            ConnectFourGame.EMPTY: "⚪",
            ConnectFourGame.RED: "🔴",
            ConnectFourGame.YELLOW: "🟡"
        }

        html = "<table style='border-collapse:collapse; text-align:center;'>"

        for r in range(game.ROWS):
            html += "<tr>"
            for c in range(game.COLS):
                html += f"<td style='font-size:24px; padding:4px;'>{symbols[game.board[r][c]]}</td>"
            html += "</tr>"

        html += "</table>"
        return html

In [34]:
class ConnectFourUI:
    def __init__(self, ai_depth=3):
        self.game = ConnectFourGame()
        self.ai_depth = ai_depth

        self.status = widgets.HTML(value="<b>RED moves first. Click a column.</b>")
        self.board_display = widgets.HTML(value=connect4_board_html(self.game))

        self.column_buttons = []

        for col in range(ConnectFourGame.COLS):
            button = widgets.Button(
                description=str(col + 1),
                layout=widgets.Layout(width="40px")
            )
            button.on_click(lambda btn, c=col: self.on_column_click(c))
            self.column_buttons.append(button)

        self.reset_button = widgets.Button(description="Reset", button_style="info")
        self.reset_button.on_click(self.on_reset)

        self.widget = widgets.VBox([
            self.status,
            self.board_display,
            widgets.HBox(self.column_buttons),
            self.reset_button
        ])

        display(self.widget)
        self.refresh()
    def refresh(self):
        self.board_display.value = connect4_board_html(self.game)

        terminal = self.game.is_terminal()

        for col, button in enumerate(self.column_buttons):
            button.disabled = terminal or col not in self.game.available_columns()

        if terminal:
            winner = self.game.get_winner()

            if winner == ConnectFourGame.RED:
                self.status.value = "<b>RED wins!</b>"
            elif winner == ConnectFourGame.YELLOW:
                self.status.value = "<b>YELLOW wins!</b>"
            else:
                self.status.value = "<b>Draw!</b>"
        else:
            player_name = "RED" if self.game.current_player == ConnectFourGame.RED else "YELLOW"
            self.status.value = f"<b>Current player: {player_name}</b>"

    def on_column_click(self, col):
        if self.game.is_terminal():
            return

        if col not in self.game.available_columns():
            return

        if self.game.current_player != ConnectFourGame.RED:
            return

        # Human RED move
        self.game.drop_piece(col)

        # AI YELLOW move
        if not self.game.is_terminal():
            _, ai_col = minimax_connect4(self.game, self.ai_depth)

            if ai_col is not None:
                self.game.drop_piece(ai_col)

        self.refresh()

    def on_reset(self, button):
        self.game = ConnectFourGame()
        self.refresh()

In [35]:
ConnectFourUI(ai_depth=4)